# Silver layer — sổ tay (SCAFFOLD, Phase 2)
Bronze ghi nguyên trạng; **Silver diễn giải**: parse → ép kiểu → mask PII → dedup → `MERGE INTO` theo primary key ra current-state. Logic thật viết ở Phase 2 sau khi brainstorm; notebook này để thăm dò trước.

In [ ]:
# --- bootstrap: cho phép import gtl_session dù notebook nằm ở thư mục con ---
import sys
from pathlib import Path

for c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
          Path.home() / "working/projects/Governed-Transaction-Lakehouse/spark"]:
    if (c / "gtl_session.py").exists():
        sys.path.insert(0, str(c))
        print("spark dir:", c)
        break
else:
    raise RuntimeError("khong tim thay gtl_session.py — chay jupyter tu trong project")

In [ ]:
# Xây SparkSession (lần đầu sẽ kéo JAR từ Maven, hơi lâu).
# Notebook chạy bằng chính venv host: ~/working/gtl-spark-venv/bin/jupyter lab
from gtl_session import CATALOG, get_spark

spark = get_spark("gtl-notebook", master="local[2]", driver_memory="2g")
spark

## Đọc Bronze làm đầu vào cho Silver

In [ ]:
from silver_transform import read_bronze

read_bronze(spark, "transactions").show(3, truncate=False)

## Nháp: current-state theo primary key (tiền thân MERGE của Phase 2)
Với mỗi `txn_id`, lấy bản CDC mới nhất theo `source_ts_ms` → trạng thái hiện tại. Phase 2 sẽ làm bằng `MERGE INTO` tăng dần thay vì quét lại toàn bộ như đây.

In [ ]:
from pyspark.sql import Window as W
from pyspark.sql import functions as F

src = f"{CATALOG}.bronze.transactions"
parsed = (spark.table(src)
    .where("op IN ('c','u','d')")
    .select(
        F.get_json_object("value", "$.after.txn_id").cast("bigint").alias("txn_id"),
        F.get_json_object("value", "$.after.status").alias("status"),
        F.col("op"), F.col("source_ts_ms")))
w = W.partitionBy("txn_id").orderBy(F.col("source_ts_ms").desc())
current = (parsed.withColumn("rn", F.row_number().over(w))
                 .where("rn = 1").drop("rn"))
current.groupBy("status").count().show()

---
`build_silver()` trong `silver_transform.py` còn là stub — sẽ hiện thực ở Phase 2.

In [ ]:
# spark.stop()